In [1]:
import sys
import numpy as np
import pybullet as p
import time
import logging
from typing import List, Tuple, Optional, Sequence, Collection, Dict, Any, cast, Union, Callable
import random
import json
import ipdb

from predicators.structs import Action, Array, GroundAtom, Object, State, Type, ParameterizedOption
from predicators import utils
from predicators.settings import CFG
from gym.spaces import Box

#Import core environment methods, robot function etc.

from predicators.envs.pybullet_blocks import PyBulletBlocksEnv
from predicators.envs.pybullet_multitable_blocks import PyBulletMultiTableBlocksEnv
from predicators.envs.pybullet_env import PyBulletEnv, create_pybullet_block
from predicators.pybullet_helpers.robots import SingleArmPyBulletRobot
from predicators.pybullet_helpers.robots.mobile_single_arm import MobileSingleArmPyBulletRobot
from predicators.pybullet_helpers.geometry import Pose
from predicators.pybullet_helpers.joint import JointPositions, get_joint_infos, get_joint_positions
from predicators.pybullet_helpers.link import get_link_state, get_link_pose

#Import the functions that are to be tested:

from predicators.pybullet_helpers.motion_planning import run_motion_planning, run_base_motion_planning,\
                                                            run_coordinated_motion_planning
#The pick/place options to be tested are accessed via the env instance
from predicators.pybullet_helpers.controllers import execute_coordinated_path, create_move_end_effector_to_pose_option,\
                                                    create_change_fingers_option, create_base_reset_based_move_base_option,\
                                                    create_arm_motion_planning_option
#Configure logging for better debugging outputs:
#logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logging.basicConfig(
    level=logging.WARNING,                    
    format="%(asctime)s %(name)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

pybullet build time: Jan 29 2025 23:16:28


In [2]:
#Defining test configuration, and overriding some default ones:
CFG.pybullet_robot = "fetch_mobile"
CFG.use_gui = True
#Draws helpful debug lines in the workspace.
#NOT SURE WHETHER TO USE THIS. WILL DECIDE AFTER A COUPLE RUNS.
#CFG.pybullet_draw_debug = True
#Initializing with standard size of blocks.
CFG.blocks_block_size = 0.05
CFG.pybullet_birrt_num_iters = 50
CFG.pybullet_birrt_num_attempts = 10
CFG.pybullet_birrt_smooth_amt = 20
CFG.seed = random.randint(0,10000)
#CFG.seed = 12
#Num of PyBullet physics steps per high-level Action in visualize_action_sequence
CFG.pybullet_sim_steps_per_action = 120

In [3]:
CFG.pybullet_robot = "fetch_mobile"


# 1. Initialize your multi-table environment  
env = PyBulletMultiTableBlocksEnv(use_gui=True, num_tables=3)
CFG.env = env.get_name()

symbolic_robot = env._robot
robot = env._pybullet_robot
physics_client_id = env._physics_client_id
assert physics_client_id is not None

In [4]:
table_configs = {  
    0: {  # Table 0: Random piles  
        'exact_state': {},  
        'setup': 'pile',  
        'params': [2, 3]  # 2 piles, 3 blocks per pile  
    },  
    1: {  # Table 1: Exact pile configuration  
        'exact_state': {  
            'pile1': ['red', 'blue', 'green'],  
            'pile2': ['yellow', 'purple']  
        },  
        'setup': 'exact_pile',  
        'params': None  
    },  
    2: {  # Table 2: Empty initially  
        'exact_state': [],  
        'setup': 'exact_scattered',  
        'params': None  
    }  
}  
  
# Create the initial state  
initial_state = env.set_state(table_configs)

In [5]:
initial_state.base_pose

(0.5464, 1.0, 0.0)

In [6]:
robot.get_base_pose(physics_client_id)

(0.55, 1.0, 0.0)

In [7]:
#Store the robot's default arm and finger joint positions.
home_arm_joints = robot.initial_joint_positions

table_obj = initial_state.get_objects(env._table_type)[0]

In [8]:
type(home_arm_joints)

list

In [9]:
table_obj

table0:table

In [10]:
robot.get_joints()

[0.5659510603602675,
 -0.7716089106205581,
 1.8670912214738085,
 -1.3866771764372758,
 0.7582547494809759,
 -1.49262709391197,
 -0.6660005202535766,
 0.03999999910593033,
 0.03999999910593033]

In [12]:
robot.move_base_to(target_pose=(-1.0649735846778243, 1.1245188158354025, 2.607610507305765), physics_client_id=physics_client_id)
robot.set_joints([0.5843417386893557, -0.6887649963243547, 1.7439425449378185, -1.3807406241274613, 0.8641376874324078, -1.5602694150920702, -0.7302861718986345, 0.040003538929260535, 0.03999627319953056])

In [14]:
target_ee_pose=Pose(position=(-1.6484019, 1.5463182, 0.5124532103538513), orientation=(0.7071, 0.0, -0.7071, 0.0))

In [17]:
target_joint_positions = robot.inverse_kinematics(target_ee_pose, validate=True, set_joints=False)

In [18]:
[0.589175615118877, -0.65250313562897, 1.6930158049465427, -1.379153834730108, 0.9088715780067638, -1.5913719074848345, -0.7556601772605692, 0.040003271556787894, 0.03999654046535791]

[-0.4895156080938867,
 0.1472935787624967,
 1.9622069558056578,
 1.0768837205648267,
 1.2457702564512336,
 -1.8366006198572606,
 3.121895462104825,
 0.04,
 0.04]

In [ ]:
[0.589175615118877, -0.65250313562897, 1.6930158049465427, -1.379153834730108, 0.9088715780067638, -1.5913719074848345, -0.7556601772605692, 0.040003271556787894, 0.03999654046535791]

In [5]:
symbolic_robot

robby:robot

In [16]:
initial_state.base_pose

In [15]:
robot_pos

In [8]:
block_obj = initial_state.get_objects(env._block_type)

In [9]:
block_to_pick_obj = block_obj[2]

In [10]:
block_to_pick_obj

block0_0_2:block

In [11]:
symbolic_robot

robby:robot

In [12]:
initial_state.get(table_obj, "pose_z")

0.0

In [13]:
home_orn = env.get_robot_ee_home_orn()
move_option_memory = {}
params_space = Box(low=np.array([], dtype=np.float32),
                  high=np.array([], dtype=np.float32), dtype=np.float32)

In [14]:
def _create_move_robot_base_option(name: str, robot: MobileSingleArmPyBulletRobot, 
                                    option_types: Sequence[Type],  params_space:Box, 
                                    env: PyBulletBlocksEnv, physics_client_id: int) -> ParameterizedOption:

        """Compute/derive values required to initialize the base motion option which first plans
        then executes the base motion via differential drive.
        """

        def get_current_base_and_arm_pose(robot: MobileSingleArmPyBulletRobot, state:State, objects: Sequence[Object],
                                         params: Array) -> Tuple[Tuple[float, float, float], JointPositions]:

            current_base_pose = robot.get_base_pose(physics_client_id)
            current_joint_positions = robot.get_joints()

            return current_base_pose, current_joint_positions

        assert physics_client_id is not None
        all_bodies = [p.getBodyUniqueId(i, physicsClientId=physics_client_id)
                for i in range(p.getNumBodies(physicsClientId=physics_client_id))]

        # ipdb.set_trace()

        collision_bodies = [b for b in all_bodies if b!=robot.robot_id and b!=0]

        held_obj_id_at_start = env._held_obj_id
        ee_link_to_held_obj = None
        if held_obj_id_at_start is not None:
            # 1. world -> base_link pose | It actually is World -> EE transform
            world_to_ee_pos, world_to_ee_orn = get_link_pose(
                                                        robot.robot_id,
                                                        robot.end_effector_id,
                                                        physics_client_id=physics_client_id
                                                    )

            # 2. base_link -> world
            ee_to_world_pos, ee_to_world_orn = p.invertTransform(
                                                        world_to_ee_pos, world_to_ee_orn
                                                    )
                                                    
            # 3. world -> object
            world_to_obj_pos, world_to_obj_orn = p.getBasePositionAndOrientation(
                                                        held_obj_id_at_start, physicsClientId=physics_client_id
                                                    )

            # 4. base_link -> object (chain transforms)
            ee_link_to_held_obj = p.multiplyTransforms(
                                                        ee_to_world_pos, ee_to_world_orn,
                                                        world_to_obj_pos, world_to_obj_orn
                                                        )
        home_orn = PyBulletBlocksEnv.get_robot_ee_home_orn()

        return create_base_reset_based_move_base_option(name=name, robot=robot, types=option_types, params_space=params_space, 
            get_current_base_and_arm_pose=get_current_base_and_arm_pose, home_orn=home_orn, collision_bodies=collision_bodies, 
            seed=CFG.seed, physics_client_id=physics_client_id, held_object_id_at_start=held_obj_id_at_start, 
            ee_to_held_object_transform_at_start=ee_link_to_held_obj)


In [15]:
def _create_move_arm_to_above_block_option(name: str, z_func: Union[Callable[[float], float], float], finger_status:str,
                                               robot: MobileSingleArmPyBulletRobot, option_types:List[Type], 
                                               params_space: Box, env: PyBulletMultiTableBlocksEnv, 
                                               physics_client_id: int) -> ParameterizedOption:

        """Compute/derive values required to initialize the arm motion option which first plans
        then executes the the arm motion.
        """

        initial_joint_position = robot.get_joints()
        held_obj_id = env._held_obj_id
        ee_link_to_held_object = None
        # if "Grasp" in name:
        #     assert held_obj_id is None, "Cannot be holding an item during Pick."
        # elif "Stack" in name:
        #     assert held_obj_id is not None, "Must be holding an item during Stack."

        if held_obj_id is not None:
            # 1. world -> base_link pose | It actually is World -> EE transform
            world_to_ee_pos, world_to_ee_orn = get_link_pose(
                                                        robot.robot_id,
                                                        robot.end_effector_id,
                                                        physics_client_id=physics_client_id
                                                    )

            # 2. base_link -> world
            ee_to_world_pos, ee_to_world_orn = p.invertTransform(
                                                        world_to_ee_pos, world_to_ee_orn
                                                    )
                                                    
            # 3. world -> object
            world_to_obj_pos, world_to_obj_orn = p.getBasePositionAndOrientation(
                                                        held_obj_id, physicsClientId=physics_client_id
                                                    )

            # 4. base_link -> object (chain transforms)
            ee_link_to_held_obj = p.multiplyTransforms(
                                                        ee_to_world_pos, ee_to_world_orn,
                                                        world_to_obj_pos, world_to_obj_orn
                                                        )
        

        all_bodies = [p.getBodyUniqueId(i, physicsClientId=physics_client_id)
                for i in range(p.getNumBodies(physicsClientId=physics_client_id))]

        collision_bodies = [b for b in all_bodies if b!=robot.robot_id and b!=0]

        home_orn = PyBulletBlocksEnv.get_robot_ee_home_orn()

        return create_arm_motion_planning_option(name=name, robot=robot, types=option_types, params_space=params_space, 
                                physics_client_id=physics_client_id,  initial_joint_positions=initial_joint_position, 
                                z_func=z_func, home_orn=home_orn, collision_bodies=collision_bodies, seed=CFG.seed, 
                                held_obj_id=held_obj_id, base_link_to_held_object=ee_link_to_held_object)


In [16]:
state = env._get_state()

In [17]:
move_option = _create_move_robot_base_option(name="MoveBase", robot=robot, option_types=[env._robot_type,env._table_type], params_space=params_space,
                                      env=env, physics_client_id=physics_client_id)

In [18]:
move_option.initiable

<function predicators.pybullet_helpers.controllers.create_base_reset_based_move_base_option.<locals>._initiable(state: predicators.structs.State, memory: dict, objs: Sequence[predicators.structs.Object], params: numpy.ndarray[typing.Any, numpy.dtype[numpy.float32]]) -> bool>

In [19]:
grounded_move = move_option.ground([symbolic_robot, table_obj], np.array([], dtype=np.float32))

In [20]:
grounded_move.initiable

<function predicators.structs.ParameterizedOption.ground.<locals>.<lambda>(s)>

In [21]:
assert grounded_move.initiable(state)

In [22]:
assert grounded_move.initiable(state)

In [23]:
print(grounded_move)

_Option(name='MoveBase', objects=[robby:robot, table0:table], params=array([], dtype=float32))


In [24]:
# assert grounded_move.initiable(move_option_memory)

# print(grounded_move)

#state = initial_state

move_action_list = []

while not grounded_move.terminal(state):
    action = grounded_move.policy(state=state)
    move_action_list.append(action)

2025-08-29 11:47:02 root [WARNING] Max time reached. No IKFast solution found.
2025-08-29 11:47:02 root [WARNING] No IK solutions found in 0.501 seconds
2025-08-29 11:47:03 root [WARNING] Max time reached. No IKFast solution found.
2025-08-29 11:47:03 root [WARNING] No IK solutions found in 0.501 seconds
2025-08-29 11:47:03 root [WARNING] Max time reached. No IKFast solution found.
2025-08-29 11:47:03 root [WARNING] No IK solutions found in 0.501 seconds



IK Succeeded for Target EE position: Pose(position=(1.0, -0.5, 0.6000000000000001), orientation=(0.7071, 0.0, -0.7071, 0.0)) at test pose: (0.31559606290566056, -0.10296737442933324, -0.5256694429708686).


In [25]:
for i, action in enumerate(move_action_list):
    print(f"\nAction idx: {i}")
    state = env.simulate(state,action)


Action idx: 0

Action idx: 1

Action idx: 2

Action idx: 3

Action idx: 4

Action idx: 5

Action idx: 6

Action idx: 7

Action idx: 8

Action idx: 9

Action idx: 10

Action idx: 11

Action idx: 12

Action idx: 13

Action idx: 14

Action idx: 15

Action idx: 16

Action idx: 17

Action idx: 18

Action idx: 19

Action idx: 20

Action idx: 21

Action idx: 22

Action idx: 23

Action idx: 24

Action idx: 25

Action idx: 26

Action idx: 27

Action idx: 28

Action idx: 29

Action idx: 30

Action idx: 31

Action idx: 32

Action idx: 33

Action idx: 34

Action idx: 35

Action idx: 36

Action idx: 37

Action idx: 38

Action idx: 39

Action idx: 40

Action idx: 41

Action idx: 42

Action idx: 43

Action idx: 44

Action idx: 45

Action idx: 46

Action idx: 47

Action idx: 48

Action idx: 49

Action idx: 50

Action idx: 51

Action idx: 52

Action idx: 53

Action idx: 54

Action idx: 55

Action idx: 56

Action idx: 57

Action idx: 58

Action idx: 59

Action idx: 60

Action idx: 61

Action idx: 62

A

In [26]:
len(move_action_list)

113

In [27]:
arm_motion_option =  _create_move_arm_to_above_block_option(name="MoveEndEffectorToPreGrasp", z_func=lambda z: (z+0.2), finger_status="open",
                                                           robot=robot, option_types=[env._robot_type,env._block_type], params_space=params_space,
                                                           env=env, physics_client_id=physics_client_id)

In [28]:
grounded_arm_motion = arm_motion_option.ground([symbolic_robot, block_to_pick_obj], np.array([], dtype=np.float32))

In [29]:
grounded_arm_motion.initiable

<function predicators.structs.ParameterizedOption.ground.<locals>.<lambda>(s)>

In [30]:
arm_motion_action_list = []

while not grounded_arm_motion.terminal(state):
    # ipdb.set_trace()
    action = grounded_arm_motion.policy(state)
    arm_motion_action_list.append(action)

> /home/cloaked04/projects/predicators/predicators/pybullet_helpers/controllers.py(1432)_plan_once_and_cache_actions()
   1431         ipdb.set_trace()
-> 1432         if "Grasp" in name or "Stack" in name:
   1433 



ipdb>  n


> /home/cloaked04/projects/predicators/predicators/pybullet_helpers/controllers.py(1434)_plan_once_and_cache_actions()
   1433 
-> 1434             _, block = objects
   1435 



ipdb>  n


> /home/cloaked04/projects/predicators/predicators/pybullet_helpers/controllers.py(1436)_plan_once_and_cache_actions()
   1435 
-> 1436             block_x, block_y, block_z = (state.get(block, "pose_x"),
   1437                                          state.get(block, "pose_y"),



...

ipdb>  n


> /home/cloaked04/projects/predicators/predicators/pybullet_helpers/controllers.py(1521)_plan_once_and_cache_actions()
   1520             # Build a full-size action array and set arm joints. Fingers included in q if you put them there.
-> 1521             arr = np.zeros_like(robot.action_space.low, dtype=np.float32)
   1522             # Place the planned arm (and fingers, if present) into arr:



ipdb>  n


> /home/cloaked04/projects/predicators/predicators/pybullet_helpers/controllers.py(1524)_plan_once_and_cache_actions()
   1523             # Assuming run_motion_planning used the same ordering/dim as robot.get_joints()
-> 1524             arr[:len(q)] = np.array(q, dtype=np.float32)
   1525             # Clip to action space just in case



ipdb>  c


In [31]:
arm_motion_action_list

[Action(_arr=array([-0.5515945 , -0.8148348 , -1.9363824 , -1.3911896 , -0.7005961 ,
        -1.4592812 ,  0.62851405,  0.03999368,  0.04000647], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (0.0, 0.0)}}),
 Action(_arr=array([-0.5430261 , -0.7776596 , -1.9063122 , -1.3363231 , -0.75421864,
        -1.470429  ,  0.5662264 ,  0.03999368,  0.04000647], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (0.0, 0.0)}}),
 Action(_arr=array([-0.53445774, -0.74048436, -1.8762422 , -1.2814567 , -0.80784124,
        -1.4815767 ,  0.5039388 ,  0.03999368,  0.04000647], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (0.0, 0.0)}}),
 Action(_arr=array([-0.52588934, -0.7033092 , -1.8461721 , -1.2265904 , -0.8614638 ,
        -1.4927244 ,  0.44165114,  0.03999368,  0.04000647], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (0.0, 0.0)}}),
 Action(_arr=array([-0.51732093, -0.666134  , -1.816102  , -

In [45]:
quat_orn = env.get_robot_ee_home_orn()
theta_orn = p.getEulerFromQuaternion(quat_orn)

In [46]:
new_pose = (0.55, 1.0, 0.0)

In [47]:
robot.robot_id

1

In [48]:
p.resetBasePositionAndOrientation(robot.robot_id, new_pose, quat_orn, physics_client_id)
robot.set_joints(home_arm_joints)